In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from joblib import dump

In [2]:
#Load and preprocess the dataset
data = pd.read_csv('Twitter_Data.csv')

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27481 entries, 0 to 27480
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   textID         27481 non-null  object
 1   text           27480 non-null  object
 2   selected_text  27480 non-null  object
 3   sentiment      27481 non-null  object
dtypes: object(4)
memory usage: 858.9+ KB


In [ ]:
# Convert 'clean_text' column to strings
data['clean_text'] = data['selected_text'].astype(str)
data['clean_text'] = data['clean_text'].str.replace('[^a-zA-Z\s]', '', regex=True).str.lower()

In [5]:
# Split data into training and testing sets
X = data['clean_text']
y = data['sentiment']

# Check unique values in the 'sentiment' column
unique_sentiments = y.unique()
print("Unique Sentiments:", unique_sentiments)

# Ensure that your labels are numeric
y = y.replace({'negative': 0, 'neutral': 1, 'positive': 2})

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Unique Sentiments: ['neutral' 'negative' 'positive']


/var/folders/52/82n4d5p91fnfw18st7nn3sn80000gn/T/ipykernel_61580/757857932.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'negative': 0, 'neutral': 1, 'positive': 2})


In [6]:
y_train.head()

11293    1
11299    1
18204    1
22728    1
1231     1
Name: sentiment, dtype: int64

In [15]:
# Tokenize and pad text sequences
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=100, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=100, padding='post')

In [16]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# One-hot encode labels
num_classes = len(unique_sentiments)
y_train_onehot = tf.keras.utils.to_categorical(y_train_encoded, num_classes=num_classes)
y_test_onehot = tf.keras.utils.to_categorical(y_test_encoded, num_classes=num_classes)

In [17]:
y_train_onehot[0] # Neg, Neu, Pos 

array([0., 1., 0.])

In [18]:
# Build a simple LSTM model with 3 output units
model = tf.keras.Sequential([
    Embedding(input_dim=10000, output_dim=100, input_length=100),
    LSTM(128),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

/opt/anaconda3/envs/glueviz/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [19]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with a reduced batch size
model.fit(X_train_pad, y_train_onehot, epochs=10, batch_size=32, validation_data=(X_test_pad, y_test_onehot))

Epoch 1/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 27s 37ms/step - accuracy: 0.4033 - loss: 1.0889 - val_accuracy: 0.4057 - val_loss: 1.0891
Epoch 2/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.4043 - loss: 1.0880 - val_accuracy: 0.4057 - val_loss: 1.0868
Epoch 3/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.4043 - loss: 1.0875 - val_accuracy: 0.4057 - val_loss: 1.0869
Epoch 4/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 26s 37ms/step - accuracy: 0.4043 - loss: 1.0874 - val_accuracy: 0.4057 - val_loss: 1.0869
Epoch 5/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.4043 - loss: 1.0873 - val_accuracy: 0.4057 - val_loss: 1.0866
Epoch 6/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.4043 - loss: 1.0873 - val_accuracy: 0.4057 - val_loss: 1.0867
Epoch 7/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 27s 39ms/step - accuracy: 0.4043 - loss: 1.0871 - val_accuracy: 0.4057 - val_loss: 1.0868
Epoch 8/10
687/687 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.4043 - loss: 1.0872 - 

In [21]:
# Save the trained model
model.save('sentiment_model.h5')
dump(tokenizer, 'tokenizer.joblib')

['tokenizer.joblib']